# 01 — Exploratory Data Analysis

**Purpose:** Understand the Credit Card Transactions Fraud Detection dataset
before any modeling decisions are made.

**Sections:**
1. Load & combine both CSV files
2. Structural inspection (shape, dtypes, head)
3. Missing values
4. Duplicates
5. Suspicious / invalid values
6. Target distribution (class imbalance)
7. Deeper EDA (distributions, fraud vs legitimate, correlations)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_DIR = Path("..") / "data" / "raw"

train = pd.read_csv(DATA_DIR / "fraudTrain.csv")
test  = pd.read_csv(DATA_DIR / "fraudTest.csv")

print("fraudTrain.csv:", train.shape)
print("fraudTest.csv :", test.shape)

# Combine — it will be done on a stratified split later

df = pd.concat([train, test], ignore_index=True)
print("Combined      :", df.shape)

# Free memory — the original frames are no longer needed
del train, test

fraudTrain.csv: (1296675, 23)
fraudTest.csv : (555719, 23)
Combined      : (1852394, 23)


In [3]:
print("Shape:", df.shape)
print()
df.info()

Shape: (1852394, 23)

<class 'pandas.DataFrame'>
RangeIndex: 1852394 entries, 0 to 1852393
Data columns (total 23 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Unnamed: 0             int64  
 1   trans_date_trans_time  str    
 2   cc_num                 int64  
 3   merchant               str    
 4   category               str    
 5   amt                    float64
 6   first                  str    
 7   last                   str    
 8   gender                 str    
 9   street                 str    
 10  city                   str    
 11  state                  str    
 12  zip                    int64  
 13  lat                    float64
 14  long                   float64
 15  city_pop               int64  
 16  job                    str    
 17  dob                    str    
 18  trans_num              str    
 19  unix_time              int64  
 20  merch_lat              float64
 21  merch_long             float64
 22  is_frau

In [4]:
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [5]:
df.dtypes.to_frame("dtype")

,dtype
Unnamed: 0,int64
trans_date_trans_time,str
cc_num,int64
merchant,str
category,str
amt,float64
first,str
last,str
gender,str
street,str


In [6]:
missing = df.isnull().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(4)
missing = missing[missing["missing_count"] > 0].sort_values("missing_count", ascending=False)
print(f"Columns with missing values: {len(missing)}")
missing

Columns with missing values: 0


,missing_count,missing_pct


In [7]:
full_dupes = df.duplicated().sum()
print("Fully duplicated rows:", full_dupes)

if "trans_num" in df.columns:
    id_dupes = df["trans_num"].duplicated().sum()
    print("Duplicated trans_num:", id_dupes)

Fully duplicated rows: 0
Duplicated trans_num: 0


In [8]:
target_col = "is_fraud"

counts = df[target_col].value_counts().sort_index()
pcts   = (df[target_col].value_counts(normalize=True).sort_index() * 100).round(4)

summary = pd.DataFrame({"count": counts, "percent": pcts})
summary.index = ["Legitimate (0)", "Fraud (1)"]
print(summary)

legit = (df[target_col] == 0).sum()
fraud = (df[target_col] == 1).sum()
print(f"\nLegitimate : Fraud ≈ {legit/fraud:.1f} : 1")

                  count  percent
Legitimate (0)  1842743   99.479
Fraud (1)          9651    0.521

Legitimate : Fraud ≈ 190.9 : 1


In [9]:
# Numeric columns — min / max / mean to spot any impossible values
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,1852394.0,5.371934e+05,3.669110e+05,0.000000e+00,2.315490e+05,4.630980e+05,8.335758e+05,1.296674e+06
cc_num,1852394.0,4.173860e+17,1.309115e+18,6.041621e+10,1.800429e+14,3.521417e+15,4.642255e+15,4.992346e+18
amt,1852394.0,7.006357e+01,1.592540e+02,1.000000e+00,9.640000e+00,4.745000e+01,8.310000e+01,2.894890e+04
zip,1852394.0,4.881326e+04,2.688185e+04,1.257000e+03,2.623700e+04,4.817400e+04,7.204200e+04,9.992100e+04
lat,1852394.0,3.853931e+01,5.071470e+00,2.002710e+01,3.466890e+01,3.935430e+01,4.194040e+01,6.669330e+01
long,1852394.0,-9.022783e+01,1.374789e+01,-1.656723e+02,-9.679800e+01,-8.747690e+01,-8.015800e+01,-6.795030e+01
city_pop,1852394.0,8.864367e+04,3.014876e+05,2.300000e+01,7.410000e+02,2.443000e+03,2.032800e+04,2.906700e+06
unix_time,1852394.0,1.358674e+09,1.819508e+07,1.325376e+09,1.343017e+09,1.357089e+09,1.374581e+09,1.388534e+09
merch_lat,1852394.0,3.853898e+01,5.105604e+00,1.902742e+01,3.474012e+01,3.936890e+01,4.195626e+01,6.751027e+01
merch_long,1852394.0,-9.022794e+01,1.375969e+01,-1.666716e+02,-9.689944e+01,-8.744069e+01,-8.024511e+01,-6.695090e+01


In [10]:
# Categorical columns — how many unique values each of them holds
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
for c in cat_cols:
    print(f"{c:25s} unique={df[c].nunique():>8,}  sample={df[c].dropna().unique()[:3]}")

C:\Users\T480S\AppData\Local\Temp\ipykernel_17456\572298659.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns.tolist()


trans_date_trans_time     unique=1,819,551  sample=<StringArray>
['2019-01-01 00:00:18', '2019-01-01 00:00:44', '2019-01-01 00:00:51']
Length: 3, dtype: str
merchant                  unique=     693  sample=<StringArray>
['fraud_Rippin, Kub and Mann', 'fraud_Heller, Gutmann and Zieme', 'fraud_Lind-Buckridge']
Length: 3, dtype: str
category                  unique=      14  sample=<StringArray>
['misc_net', 'grocery_pos', 'entertainment']
Length: 3, dtype: str
first                     unique=     355  sample=<StringArray>
['Jennifer', 'Stephanie', 'Edward']
Length: 3, dtype: str
last                      unique=     486  sample=<StringArray>
['Banks', 'Gill', 'Sanchez']
Length: 3, dtype: str
gender                    unique=       2  sample=<StringArray>
['F', 'M']
Length: 2, dtype: str
street                    unique=     999  sample=<StringArray>
['561 Perry Cove', '43039 Riley Greens Suite 393', '594 White Dale Suite 530']
Length: 3, dtype: str
city                      unique=    

## Observations from Initial Inspection

Alright, first pass done. Here's what the data looks like after loading and combining both CSVs.

### Size & Shape
- Combined dataset: **1,852,394 rows × 23 columns** (1.29M from fraudTrain + 555K from fraudTest).
- In-memory footprint: **~325 MB**.
- Every transaction has a unique `trans_num`, and there are zero full-row duplicates. So the dataset is clean on the duplication front.

### Missing Data
- **Zero missing values** in any column. Nothing to impute.
- Caveat: no NaNs doesn't automatically mean "complete." A value could be encoded as an empty string or a placeholder like -1 and still show up as "present." I checked the numeric ranges and nothing jumped out as a disguised null, but keeping an eye on this.

### The Big One — Class Imbalance
- **1,842,743 legitimate** (99.479%)
- **9,651 fraudulent** (0.521%)
- Ratio: **~191 : 1**

This is the defining feature of the dataset. A model that just predicts "not fraud" for everything would score **99.48% accuracy** while catching **zero fraud**. So accuracy is basically useless as a headline metric here. The metrics that matter for this project will be
**Precision, Recall, F1, and Average Precision (AP)**.

### Column Types
- 5 × float64 (`amt`, `lat`, `long`, `merch_lat`, `merch_long`)
- 6 × int64 (including `is_fraud`, our target)
- 12 × str (text columns)

Two columns need conversion before modeling:
- `trans_date_trans_time` — currently a string, should be datetime.
- `dob` — same story, and it's also PII.

### Column-Level Notes

**Junk column to drop:**
- `Unnamed: 0` — this is the row index from the original CSVs that got
  written into the file when they were saved. Its max value is only
  1,296,674 while we have 1,852,394 rows, which confirms it's not even a
  valid counter anymore after combining the two files. Drop it.

**PII to drop (no reason for a fraud model to see names/addresses):**
- `cc_num`, `first`, `last`, `street`, `city`, `zip`, `dob`, `trans_num`.

**Categorical columns — cardinality matters for encoding later:**
- Easy one-hot: `gender` (2 values), `category` (14 values).
- Medium cardinality, decision needed: `state` (51), `merchant` (693),`city` (906), `street` (999). One-hot would blow up the feature space for these.
- Never one-hot: `trans_date_trans_time` — 1.8M unique values.

**Numeric ranges — all sane:**
- `amt`: $1.00 to $28,948.90 — no negatives, no zeros, no absurd maxes.
- `lat` / `long` / `merch_lat` / `merch_long`: all within US bounds.
- `city_pop`: 23 to 2.9M — small towns to NYC.
- `unix_time`: ~2019 to ~2020.

### Things That Looked Suspicious But Aren't
- Every merchant name starts with `fraud_` — e.g. `fraud_Rippin, Kub and Mann`. This looked like a leak at first glance, but it's a naming convention in the synthetic data. Verified: it appears on merchants serving both fraud and non-fraud transactions, so it's not a leak.

### Misc
- Pandas 3.0 threw a deprecation warning when I filtered text columns with `include=["object"]`. It still works for now, but pandas 4.0 will require `include=["str", "string"]`. Just noting it here so future-me isn't confused when it becomes an error.
- The combined DataFrame's index is unique (thanks to `ignore_index=True` on concat), so `Unnamed: 0` is the only column with non-unique values is another reason it's useless.

### Next Up
Now that the data is inspected, the next notebook step is deeper EDA — distributions, fraud vs legitimate comparisons, correlations, and a first look at feature relationships. Then preprocessing.

In [11]:
import subprocess
for f in ["data/raw/fraudTrain.csv", "data/raw/fraudTest.csv"]:
    r = subprocess.run(["git", "check-ignore", "-v", f], capture_output=True, text=True)
    print(f"{f} → {r.stdout.strip() or 'NOT IGNORED ⚠️'}")

data/raw/fraudTrain.csv → .gitignore:4:*.csv	data/raw/fraudTrain.csv
data/raw/fraudTest.csv → .gitignore:4:*.csv	data/raw/fraudTest.csv
